# Download libraries

In [ ]:
!pip install -U transformers huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 27.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.47.1
    Uninstalling transformers-4.47.1:
      Successfully uninstalled transformers-4.47.1


In [ ]:
from huggingface_hub import login
from google.colab import userdata
data = userdata.get('HF_TOKEN')

login(data)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time

# Caricamento del modello
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b-it",
    device_map="auto",
    torch_dtype=torch.float16,  # Usa float16 per meno memoria
)

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

# Costants

In [ ]:
MODE = "train" # o validation

PATH_VALIDATION = "/content/annotations_rand_val.json"
PATH_TRAINING = "/content/annotations_train.json"
DEFAULT_PROMPT = "Given the following narrations describing actions performed by a person in a video, generate a single simple query that focuses on what the person does or interacts with. The query should capture the main activity or interaction that is described in the narrations:"
GOOGLE_DRIVE_SAVE_PATH =  "/content/gdrive/MyDrive/AML/Codice/Dati/saves_gemma/"
COLAB_PATH_SAVE = "/content/saves/"

# Request drive access to save future files

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
!mkdir -p $COLAB_PATH_SAVE

Mounted at /content/gdrive


In [ ]:
def try_split_query(query):
    possible_starting_words = ["What", "Who", "Where", "When", "Why", "How", "Which", "Whose", "Whom"]
    for word in possible_starting_words:
        if word in query:
            return word + query.split(word)[1]
    return query

In [ ]:
def generate_questions(prompt, narration_count, narrations):
  """
    Generate questions using GEMMA 2
  """
    # numero di token per narration
  tokens_per_narration = 5
  max_tokens = 1024
  input_tokens = narration_count * tokens_per_narration
  max_output_tokens = max_tokens - input_tokens
  max_output_tokens = max(0, max_output_tokens)

  input_ids = tokenizer(prompt, return_tensors="pt").to("cuda")
  outputs = model.generate(**input_ids, max_new_tokens=max_output_tokens)
  query = tokenizer.decode(outputs[0], skip_special_tokens=True)

  #Replace the input with nothing
  query = query.replace(prompt, "").strip()

  #Remove any trace of narrations from the query
  for narration in narrations:
    query = query.replace(narration, "").strip()

  #Remove usual placeholder it use
  query = query.replace("**Query:**", "").strip()

  #Get only the content before the ?
  if "**" in query:
    query = query.split("**")[1]
  if "?" in query:
    query = query.split("?")[0]
  query = query + "?"

  query = try_split_query(query)

  return query

## For every clips generate the question.

In [ ]:
def populate_file(videos):
  """
    For every clips generate the question

    Args:
      videos: list of videos
    Returns:
      None
  """
  count = 0  # Counter to know how many videos have been processed
  skipped = False
  for video in videos:
    if count >= 150 and count <= 175:
      for clip in video["clips"]:
          for language_query in clip["annotations"]:  # Access language queries for each clip
              for narrations in language_query["language_queries"]:
                  if len(narrations["narrations"]) > 0:
                    if "query" not in narrations:
                      annotations_str = "\n".join(narrations["narrations"])  # Uniamo tutte le narrazioni in un'unica stringa
                      prompt = DEFAULT_PROMPT + " " + annotations_str
                      query = generate_questions(prompt, len(narrations["narrations"]), narrations["narrations"])  # Generate questions using Gemma
                      narrations["query"] = query  # Store generated query
    else:
      skipped = True
      print("Skipping video", count, "/", len(videos))
    count += 1

    if not skipped:
      print("Video analyzed successfully", count, "/", len(videos))

      if MODE == "train":
        path_drive = GOOGLE_DRIVE_SAVE_PATH + "annotations_train_"
        path_colab = COLAB_PATH_SAVE + "annotations_train_"
      else:
        path_drive = GOOGLE_DRIVE_SAVE_PATH + "annotations_val_"
        path_colab = COLAB_PATH_SAVE + "annotations_val_"

      path_drive += str(count) + "_" + str(len(videos)) + ".json"
      path_colab += str(count) + "_" + str(len(videos)) + ".json"

      with open(path_drive, 'w') as f:
        json.dump(data, f, indent=4)
        print("Saved in " + path_drive)

      with open(path_colab, 'w') as f:
        json.dump(data, f, indent=4)
        print("Saved in " + path_colab)

    skipped = False

# Load file with questions

In [ ]:
if MODE == "train":
  !gdown --id 19KepiFNBiWuC7dHm8euOYh64VRdw99Pg
  PATH = PATH_TRAINING
else:
  !gdown --id 1si73qWWqw6oGKvD1L0GSnLKbhdaYqqjb
  PATH = PATH_VALIDATION



/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=19KepiFNBiWuC7dHm8euOYh64VRdw99Pg
From (redirected): https://drive.google.com/uc?id=19KepiFNBiWuC7dHm8euOYh64VRdw99Pg&confirm=t&uuid=4d05cbc8-6e41-4619-89c0-9fcf37ab645b
To: /content/annotations_train.json
100% 733M/733M [00:12<00:00, 59.6MB/s]


In [ ]:
# prompt: Load file json
import json
with open(PATH, 'r') as f:
  data = json.load(f)

In [ ]:
populate_file(data["videos"])

Streaming output truncated to the last 5000 lines.

Query:  What is the person doing with the hedge shear?

---------------------

Query:  What is the person doing with the hedge shear?

---------------------

Query:  picking leaves and fruit from a tree?

---------------------

Query:  #C C throws a brick at a target 

These narrations describe different actions performed by the person using a hedge shear. The query should focus on the main activity, which is cutting, dragging, picking, and throwing.?

---------------------

Query:  #C C picks a flower from the garden

These narrations describe different actions performed by the person in the video. The query should focus on what they do, rather than what they say or how they say it.?

---------------------

Query:  What is C doing with the hedge shear?

---------------------

Query:  What action is performed by C in the video?

---------------------

Query:  What action is performed by C on the tree?

---------------------

Query:  W

KeyboardInterrupt: 